# ThreatLens AI — Website Intelligence & Threat Classification
**IBM SkillsBuild Cyber Security BootCamp 2026 project submission**

This notebook is an educational cybersecurity investigation tool. It can:
1. Inspect the publicly accessible text and metadata of a website.
2. Query an existing VirusTotal v3 report for a URL, domain, IPv4 address, or file hash.
3. Display provider-reported threat labels, categories, and engine detections when available.

> **Important:** This notebook does not execute downloaded files, perform forensic malware analysis, or guarantee a website is safe. VirusTotal results may be incomplete or contain false positives. Keep API keys private.

## 1. Setup
Install dependencies from `requirements.txt`. Configure your VirusTotal API key as an environment variable named `VIRUSTOTAL_API_KEY`. Do not paste your key into a notebook cell or commit it to GitHub.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os
import re
import json
import ipaddress
from urllib.parse import urlparse
from getpass import getpass

import requests
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

VT_API_KEY = os.getenv("VIRUSTOTAL_API_KEY", "").strip()
if not VT_API_KEY:
    print("VirusTotal API key not found in environment.")
    print("For this notebook session only, enter it below if you have one.")
    VT_API_KEY = getpass("VirusTotal API key (input hidden): ").strip()

VT_BASE = "https://www.virustotal.com/api/v3"
SESSION = requests.Session()
SESSION.headers.update({"accept": "application/json"})
print("Key configured:", bool(VT_API_KEY), "(key value is never displayed)")

## 2. Validate and classify an indicator
Supported indicators are a public URL, domain, IPv4 address, or MD5/SHA-1/SHA-256 file hash. This validation only checks format; it does not prove the indicator is malicious.

In [ ]:
def classify_indicator(indicator: str):
    value = indicator.strip()
    if not value:
        raise ValueError("Enter a URL, domain, IPv4 address, or file hash.")
    if re.fullmatch(r"[a-fA-F0-9]{32}|[a-fA-F0-9]{40}|[a-fA-F0-9]{64}", value):
        return value.lower(), "file"
    try:
        ip = ipaddress.ip_address(value)
        if ip.version == 4:
            return str(ip), "ip_addresses"
        raise ValueError("This notebook currently supports IPv4 addresses only.")
    except ValueError as e:
        if "supports IPv4" in str(e):
            raise
    candidate = value if "://" in value else "https://" + value
    parsed = urlparse(candidate)
    host = (parsed.hostname or "").lower().rstrip(".")
    if parsed.scheme in ("http", "https") and host and "." in host and " " not in host:
        if "://" in value or "/" in value or "?" in value or "#" in value:
            return candidate, "url"
        return host, "domains"
    raise ValueError("Invalid indicator format. Use a public URL, domain, IPv4 address, or 32/40/64-character hex hash.")

def vt_url_id(url: str) -> str:
    import base64
    return base64.urlsafe_b64encode(url.encode()).decode().rstrip("=")

def vt_report_endpoint(indicator: str):
    normalized, kind = classify_indicator(indicator)
    if kind == "url":
        return normalized, "urls/" + vt_url_id(normalized), "URL"
    if kind == "domains":
        return normalized, "domains/" + normalized, "Domain"
    if kind == "ip_addresses":
        return normalized, "ip_addresses/" + normalized, "IPv4 address"
    return normalized, "files/" + normalized, "File hash"

print("Examples:")
for sample in ["https://example.com", "example.com", "8.8.8.8", "44d88612fea8a8f36de82e1278abb02f"]:
    try:
        print(sample, "->", classify_indicator(sample)[1])
    except ValueError as exc:
        print(sample, "->", exc)

## 3. Query VirusTotal
This requests an **existing report** from VirusTotal. It does not upload files or force a new scan. If the report is missing, the API key is invalid, or the quota is exceeded, the function reports that clearly.

In [ ]:
def lookup_virustotal(indicator: str):
    if not VT_API_KEY:
        raise RuntimeError("No VirusTotal API key configured. Set VIRUSTOTAL_API_KEY or enter it in the setup cell.")
    normalized, endpoint, kind = vt_report_endpoint(indicator)
    response = SESSION.get(
        f"{VT_BASE}/{endpoint}",
        headers={"x-apikey": VT_API_KEY},
        timeout=25,
    )
    if response.status_code == 404:
        return {"status": "not_found", "indicator": normalized, "kind": kind,
                "message": "No existing VirusTotal report was found. This does not mean the indicator is safe."}
    if response.status_code in (401, 403):
        return {"status": "auth_error", "indicator": normalized, "kind": kind,
                "message": "VirusTotal rejected the key or access is not permitted."}
    if response.status_code == 429:
        return {"status": "rate_limited", "indicator": normalized, "kind": kind,
                "message": "VirusTotal rate limit reached. Try again later."}
    response.raise_for_status()
    payload = response.json()
    attrs = payload.get("data", {}).get("attributes", {})
    stats = attrs.get("last_analysis_stats", {})
    results = attrs.get("last_analysis_results", {})
    classification = attrs.get("popular_threat_classification", {})
    detections = [
        {"engine": item.get("engine_name", "Unknown engine"), "label": item.get("result", "Malicious")}
        for item in results.values()
        if item.get("category") == "malicious" and item.get("result")
    ]
    family_names = [item.get("value") for item in classification.get("popular_threat_name", []) if item.get("value")]
    categories = [item.get("value") for item in classification.get("popular_threat_category", []) if item.get("value")]
    resource_type = {"URL":"url", "Domain":"domain", "IPv4 address":"ip-address", "File hash":"file"}[kind]
    return {
        "status": "complete", "indicator": normalized, "kind": kind,
        "source": "VirusTotal", "stats": {k: stats.get(k, 0) for k in ("malicious","suspicious","harmless","undetected","timeout")},
        "threat_label": classification.get("suggested_threat_label"),
        "threat_names": family_names[:10], "threat_categories": categories[:10],
        "engine_detections": detections[:30],
        "file_name": attrs.get("meaningful_name") or next(iter(attrs.get("names", [])), None),
        "file_type": attrs.get("type_description"),
        "last_analysis_date": attrs.get("last_analysis_date"),
        "report_url": f"https://www.virustotal.com/gui/{resource_type}/{normalized}",
        "message": "Provider-reported results only. Engine labels may disagree or be false positives."
    }

def show_vt_report(report):
    display(Markdown(f"## VirusTotal report: `{report.get('indicator','')}`"))
    display(Markdown(f"**Status:** `{report['status']}`  \n{report.get('message','')}"))
    if report["status"] != "complete":
        return
    stats = report.get("stats", {})
    display(Markdown(
        f"**Malicious:** {stats.get('malicious',0)} · "
        f"**Suspicious:** {stats.get('suspicious',0)} · "
        f"**Harmless:** {stats.get('harmless',0)} · "
        f"**Undetected:** {stats.get('undetected',0)}"
    ))
    display(Markdown(f"**Suggested threat label:** {report.get('threat_label') or 'Not provided in this report'}"))
    display(Markdown(f"**Threat/family names:** {', '.join(report.get('threat_names') or []) or 'Not provided'}"))
    display(Markdown(f"**Threat categories:** {', '.join(report.get('threat_categories') or []) or 'Not provided'}"))
    if report.get("file_name") or report.get("file_type"):
        display(Markdown(f"**File name/type:** {report.get('file_name') or 'N/A'} / {report.get('file_type') or 'N/A'}"))
    if report.get("engine_detections"):
        display(Markdown("### Security-engine detections"))
        display(report["engine_detections"])
    else:
        display(Markdown("No malicious engine labels were returned in the fields available to this notebook. This is not a guarantee of safety."))
    display(Markdown(f"[Open VirusTotal report]({report.get('report_url','')})"))

# Run with your own test indicator, e.g.:
# report = lookup_virustotal("example.com")
# show_vt_report(report)

## 4. Website intelligence: accessible page content
This function reads basic HTML metadata and visible text from a public website. It does not certify trustworthiness. Some sites block automated access, depend on JavaScript, or have terms that restrict scraping. Use only where permitted.

In [ ]:
def inspect_website(url: str, max_chars: int = 5000):
    candidate = url.strip()
    if not candidate:
        raise ValueError("Enter a website URL.")
    if "://" not in candidate:
        candidate = "https://" + candidate
    parsed = urlparse(candidate)
    if parsed.scheme not in ("http", "https") or not parsed.hostname:
        raise ValueError("Enter a valid http/https website URL.")
    host = parsed.hostname.lower()
    if host in {"localhost", "127.0.0.1", "::1"}:
        raise ValueError("Local addresses are not accepted.")
    response = requests.get(
        candidate,
        headers={"User-Agent": "ThreatLensAI-Educational-Research/1.0"},
        timeout=15,
        allow_redirects=True,
    )
    response.raise_for_status()
    content_type = response.headers.get("content-type", "")
    if "html" not in content_type.lower():
        raise ValueError(f"The response is not HTML (content type: {content_type or 'unknown'}).")
    soup = BeautifulSoup(response.text, "html.parser")
    for element in soup(["script", "style", "noscript", "svg"]):
        element.decompose()
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    description_tag = soup.find("meta", attrs={"name": re.compile("^description$", re.I)})
    description = description_tag.get("content", "").strip() if description_tag else ""
    headings = [h.get_text(" ", strip=True) for h in soup.find_all(["h1","h2"])[:12]]
    text = " ".join(soup.stripped_strings)
    text = re.sub(r"\s+", " ", text)[:max_chars]
    result = {
        "requested_url": candidate,
        "final_url": response.url,
        "http_status": response.status_code,
        "page_title": title or "(No title found)",
        "meta_description": description or "(No meta description found)",
        "headings": headings,
        "visible_text_excerpt": text,
        "note": "Page content and metadata are observations, not proof of legitimacy or safety."
    }
    display(Markdown(f"## Website overview: {result['page_title']}"))
    display(Markdown(f"- **Final URL:** {result['final_url']}\n- **HTTP status:** {result['http_status']}\n- **Description:** {result['meta_description']}"))
    if headings:
        display(Markdown("**Main headings:** " + " · ".join(headings)))
    display(Markdown("**Visible text excerpt:**\n\n" + (text or "(No visible text extracted)")))
    display(Markdown(f"> {result['note']}"))
    return result

# Example:
# site = inspect_website("https://example.com")

## 5. Combined investigation
Run a website content inspection and a separate VirusTotal reputation lookup. A website summary and a threat verdict answer different questions.

In [ ]:
def investigate_website(url: str, check_reputation: bool = True):
    site = inspect_website(url)
    vt = None
    if check_reputation:
        try:
            vt = lookup_virustotal(site["final_url"])
            show_vt_report(vt)
        except Exception as exc:
            print("Reputation lookup could not be completed:", str(exc))
    return {"website": site, "virustotal": vt}

# Uncomment and edit:
# investigation = investigate_website("https://example.com")

## 6. Save a JSON investigation report
The report is written locally to this notebook's working directory. Review it before sharing; it may contain URLs and page text.

In [ ]:
def save_report(report: dict, filename: str = "threatlens_report.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False, default=str)
    print(f"Saved report to {filename}")

# Example after running an investigation:
# save_report(investigation)

## 7. Limitations, privacy, and responsible use
- This is a learning prototype, not an enterprise security product or forensic malware sandbox.
- VirusTotal results reflect its available report and participating engines at the time of analysis.
- No report, zero detections, or an unavailable result must not be interpreted as proof that something is safe.
- Do not upload confidential or sensitive files to third-party services without authorization.
- Never publish API keys. Use environment variables or secret management.
- Inspect only systems and websites you are authorized to investigate.